In [ ]:
import os
import glob
import re
import numpy as np
import pandas as pd
import geopandas as gpd
import rioxarray as rxr
import rasterio
from rasterio.features import geometry_mask
from tqdm import tqdm

In [ ]:
# Folder containing ET tiffs
et_folder = "/Users/sidchaudhary/Downloads/OpenET_Monthly_Full_Extent"

# Shapefile path
shp_path = "/Users/sidchaudhary/Downloads/Field_Boundaries/cafe_field_perisitance_stats.shp"

# Output CSV
output_csv = "/Users/sidchaudhary/Downloads/pixel_level_ET_timeseries.csv"

In [ ]:
fields = gpd.read_file(shp_path)

print("CRS:", fields.crs)
print("Number of fields:", len(fields))
fields.head()

In [ ]:
# Only months 04,05,06,10
valid_months = ["04", "05", "06", "10"]

all_tiffs = sorted(glob.glob(os.path.join(et_folder, "ET_Monthly_*.tif")))

filtered_tiffs = [
    f for f in all_tiffs 
    if re.search(r'_(\d{4})_(\d{2})\.tif$', f).group(2) in valid_months
]

print("Total files found:", len(filtered_tiffs))

In [ ]:
records = []

for tif_path in tqdm(filtered_tiffs):

    # Extract year and month from filename
    match = re.search(r'ET_Monthly_(\d{4})_(\d{2})\.tif$', os.path.basename(tif_path))
    year, month = match.groups()
    
    # Open raster
    raster = rxr.open_rasterio(tif_path, masked=True).squeeze()
    
    # Reproject fields if CRS differs
    if fields.crs != raster.rio.crs:
        fields_proj = fields.to_crs(raster.rio.crs)
    else:
        fields_proj = fields

    # Loop over each field
    for idx, field in fields_proj.iterrows():
        
        field_geom = [field.geometry]
        
        try:
            # Clip raster to field with all_touched=True to include edge pixels
            clipped = raster.rio.clip(field_geom, fields_proj.crs, drop=True, all_touched=True)
        except Exception as e:
            # Skip if no overlap between field and raster
            continue
        
        if clipped.size == 0:
            continue
        
        et_values = clipped.values.flatten()
        et_values = et_values[~np.isnan(et_values)]
        
        if len(et_values) == 0:
            continue
        
        field_mean = np.mean(et_values)
        
        # Get pixel coordinates
        xs, ys = np.meshgrid(clipped.x.values, clipped.y.values)
        xs = xs.flatten()
        ys = ys.flatten()
        
        valid_mask = ~np.isnan(clipped.values.flatten())
        
        xs = xs[valid_mask]
        ys = ys[valid_mask]
        et_values = et_values
        
        relative_et = et_values / field_mean
        
        for x, y, et, rel in zip(xs, ys, et_values, relative_et):
            records.append({
                "field_id": idx,
                "year": int(year),
                "month": int(month),
                "x": x,
                "y": y,
                "ET": float(et),
                "relative_ET": float(rel)
            })

In [ ]:
df = pd.DataFrame(records)

print("Total rows:", len(df))
df.head()

In [ ]:
df.to_csv(output_csv, index=False)
print("Saved to:", output_csv)